# 2. PIM and Conditional Access - Implementation

SC-900 taught you *what* Privileged Identity Management (PIM) and Conditional Access (CA) are.
AZ-500 expects you to *configure* them.

## Before you run this notebook

1. Run `uv sync` in the lab folder.
2. In VS Code, pick the `.venv` kernel in the top-right kernel picker.
3. Reload the window (`Cmd+Shift+P` then *Reload Window*) if the kernel doesn't appear.

No Docker required - everything runs as plain Python.

## PIM - why standing access is dangerous

Most compromises we read about begin with **standing privileged access**: a Global Admin account that's always on, always valid, always a juicy target. PIM flips this: admins are **eligible** for a role and must **activate** it for a short time, with MFA and a justification.

### Bad to Best


In [ ]:
scenarios = [
    ('BAD',  'Alice is a permanent Global Administrator.',
     '1 compromised password => full tenant takeover, and nobody notices until audit.'),
    ('OKAY', 'Alice is Global Administrator, but her account is MFA-enforced.',
     'Much better, but her credentials are valuable 24/7 and phishing can still bypass MFA.'),
    ('BEST', 'Alice is PIM-eligible for Global Administrator, max 2h activation, MFA + approval + justification.',
     'Most of the time she is a regular user. Attackers steal nothing valuable unless she is actively activating.'),
]
for label, setup, impact in scenarios:
    print(f'{label:<5}  {setup}')
    print(f'        -> {impact}\n')


### Eligible vs active - the vocabulary the exam uses

PIM crosses two independent axes. Know all four boxes:

| | **Eligible** (must activate first) | **Active** (usable right now) |
|-|------------------------------------|-------------------------------|
| **Permanent** | *permanent eligible* — can always activate | *permanent active* — classic standing access, the thing PIM exists to remove |
| **Time-bound** | *time-bound eligible* — can activate only between a start and end date | *time-bound active* — holds the role outright, but it expires |

There is **no difference in the permissions granted** by an eligible-then-activated
assignment and a permanently active one. The only difference is *how long the account is
a target*. Terminology: a user with an active assignment is **assigned**; an eligible user
who has completed activation is **activated**. **Activation** is the eligible → active
transition, and it is the only step that can require MFA, justification or approval —
an active assignment requires nothing.

### PIM settings you must configure

For each role, PIM has per-role settings:

| Setting | Options | Typical default |
|---------|---------|-----------------|
| **Maximum activation duration** | 1 to 24 hours | 8 hours |
| **Require MFA on activation** | Yes / No | Yes for privileged roles |
| **Require justification** | Yes / No | Yes |
| **Require approval** | Yes / No + who approves | Yes for Global Admin |
| **Require ticket info** | Yes / No | No |
| **Allow permanent eligible** | Yes / No | No (set expiration) |
| **Allow permanent active** | Yes / No | No |
| **Notification on activation** | Email to admins | Yes |


In [ ]:
import json
from datetime import datetime, timedelta

PIM_ROLE_SETTINGS = {
    'Global Administrator': {
        'max_activation_hours': 2,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': True,
        'approvers': ['security-team@contoso.com'],
        'eligible_max_months': 6,
        'permanent_eligible': False,
    },
    'Contributor': {
        'max_activation_hours': 8,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': False,
    },
    'Security Reader': {
        'max_activation_hours': 8,
        'require_mfa': False,
        'require_justification': False,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': True,
    },
}

def activate_pim_role(user, role, justification, mfa, requested_hours=None, ticket=''):
    """Model the eligible -> active transition. An ACTIVE assignment would skip all
    of this; only activation gets to demand MFA, justification and approval."""
    s = PIM_ROLE_SETTINGS.get(role)
    if not s:
        return {'status': 'error', 'reason': f'Role {role} not found in PIM'}

    # The requester picks a duration; PIM caps it at the role's configured maximum
    # (the portal slider only goes from 1 hour to that maximum).
    hours = s['max_activation_hours'] if requested_hours is None else requested_hours

    failed = []
    if s['require_mfa'] and not mfa:
        failed.append('MFA required but not presented')
    if s['require_justification'] and not justification:
        failed.append('Justification required')
    if hours > s['max_activation_hours']:
        failed.append(f'Requested {hours}h exceeds the {s["max_activation_hours"]}h '
                      f'maximum activation duration for {role}')
    if failed:
        return {'status': 'DENIED', 'failed_checks': failed}

    expires = (datetime.now() + timedelta(hours=hours)).strftime('%Y-%m-%d %H:%M')
    if s['require_approval']:
        return {
            'status': 'PENDING APPROVAL',
            'role': role, 'user': user,
            'justification': justification,
            'approvers': s['approvers'],
            'would_expire': expires,
        }
    return {
        'status': 'ACTIVATED',
        'role': role, 'user': user,
        'active_until': expires,
    }

print('=== PIM activation scenarios ===\n')
print('--- 1. Global Admin (requires approval) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', 'Emergency CA policy fix', True), indent=2))

print('\n--- 2. Contributor (no approval, just MFA + justification) ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix to prod', True), indent=2))

print('\n--- 3. Contributor without MFA (blocked) ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix', False), indent=2))

print('\n--- 4. Global Admin missing justification (blocked) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', '', True), indent=2))

print('\n--- 5. Global Admin asking for a full working day (blocked) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', 'Long migration', True,
                                   requested_hours=8), indent=2))

# The whole point of PIM is that activation can FAIL. Pin that.
assert activate_pim_role('bob', 'Contributor', 'Deploy hotfix', False)['status'] == 'DENIED'
assert activate_pim_role('alice', 'Global Administrator', '', True)['status'] == 'DENIED'
assert activate_pim_role('alice', 'Global Administrator', 'x', True,
                         requested_hours=8)['status'] == 'DENIED', \
    'activation must never outlast the role\'s maximum activation duration'
# ...and that approval, not activation, is the last gate for Global Admin.
assert activate_pim_role('alice', 'Global Administrator', 'Emergency CA policy fix',
                         True)['status'] == 'PENDING APPROVAL'
assert activate_pim_role('bob', 'Contributor', 'Deploy hotfix to prod',
                         True)['status'] == 'ACTIVATED'
print('\nAll PIM activation invariants hold.')


### Activating a PIM role via the Microsoft Graph API

```bash
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/roleManagement/directory/roleAssignmentScheduleRequests' \
  --body '{
    "action": "selfActivate",
    "justification": "Emergency CA policy fix",
    "roleDefinitionId": "<role-id>",
    "directoryScopeId": "/",
    "principalId": "<user-id>",
    "scheduleInfo": {
      "startDateTime": "2026-04-17T10:00:00Z",
      "expiration": {"type": "afterDuration", "duration": "PT2H"}
    }
  }'
```

---
## Conditional Access - policy structure

Every CA policy has three parts:

1. **Assignments** - *who* (users/groups), *what* (apps), *where* (conditions like IP, device, risk).
2. **Access controls** - grant / block, plus requirements (MFA, compliant device, approved app).
3. **Session controls** - sign-in frequency, persistent browser, app-enforced restrictions.


In [ ]:
CA_POLICIES = [
    {
        'name': 'Require MFA for admins',
        'state': 'enabled',
        'assignments': {
            'users': {'include_roles': ['Global Administrator', 'Security Administrator', 'Exchange Administrator'],
                      'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa']},
        'scenario': 'Baseline: admins always need MFA. Break-glass accounts are excluded.',
    },
    {
        'name': 'Block legacy authentication',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'client_apps': ['exchangeActiveSync', 'other']},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Legacy protocols (IMAP/POP3/SMTP) cannot do MFA. Block them.',
    },
    {
        'name': 'Require compliant device for Azure portal',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': ['Microsoft Azure Management']},
        },
        'grant': {'operator': 'AND', 'controls': ['compliant_device']},
        'scenario': 'Only Intune-managed devices can reach the Azure portal.',
    },
    {
        'name': 'Require MFA for risky sign-ins',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'sign_in_risk': ['medium', 'high']},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa']},
        'scenario': 'SIGN-IN risk asks "is this login suspicious?". Step up to MFA.',
    },
    {
        'name': 'Require password change for high user risk',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},          # must be ALL resources - see the check below
            'conditions': {'user_risk': ['high']},
        },
        'grant': {'operator': 'AND', 'controls': ['password_change']},
        'scenario': 'USER risk asks "is this ACCOUNT compromised?" (leaked credentials). '
                    'Force a secure password change.',
    },
    {
        'name': 'Block access from untrusted countries',
        'state': 'report-only',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'locations': {'include': 'all', 'exclude': ['named:TrustedCountries']}},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Start in report-only, review sign-in logs, then enable.',
    },
]

print('=== Conditional Access policies ===\n')
for p in CA_POLICIES:
    state_tag = {'enabled': '[ON]', 'report-only': '[REPORT]', 'disabled': '[OFF]'}[p['state']]
    print(f'{state_tag:<10} {p["name"]}')
    print(f'   Scenario: {p["scenario"]}')
    print(f'   Grant:    {p["grant"]["operator"]} -> {p["grant"]["controls"]}\n')

# Three rules the portal enforces on "Require password change", and that exam questions
# are built out of. Getting these wrong is the classic risk-policy mistake: password
# change belongs to USER risk, MFA belongs to SIGN-IN risk.
for p in CA_POLICIES:
    if 'password_change' not in p['grant']['controls']:
        continue
    conditions = p['assignments'].get('conditions', {})
    assert p['grant']['controls'] == ['password_change'], \
        f'{p["name"]}: password change cannot be combined with other grant controls'
    assert 'user_risk' in conditions, \
        f'{p["name"]}: password change requires a USER-risk condition, not sign-in risk'
    assert p['assignments']['apps']['include'] == 'all', \
        f'{p["name"]}: password change policies must target all resources'
print('Risk-policy shape checks passed.')


## Break-glass accounts - the escape hatch

If Conditional Access breaks (misconfigured policy, MFA provider outage, expired certificate), you can lock *everyone* out, including yourself. To prevent that, create **two emergency access accounts** ("break-glass"):

| Setting | Value |
|---------|-------|
| Count | 2, minimum |
| Type | Cloud-only accounts, not synced from AD |
| Username | Not using domain names someone would guess |
| MFA | FIDO2 security key stored in a safe - **not** phone-based |
| CA policies | **Excluded from all of them** (including MFA policies!) |
| Monitoring | Alert on any sign-in using these accounts |

## How Conditional Access actually decides

Before simulating the outage, get the evaluation model right, because "the most
restrictive policy wins" is a slogan, not the algorithm. Microsoft documents two phases:

- **Phase 1 — collect session details.** User, app, IP, device state, risk. *Enabled* and
  *report-only* policies both take part; report-only stops here, which is why it can log
  "would have blocked" without blocking.
- **Phase 2 — enforcement.** Only enabled policies. **Every** applicable policy must be
  satisfied — assignments are combined with AND, and multiple policies are cumulative.
  - If any applicable policy carries the **block** control, enforcement stops right there.
    Block beats every grant, even grants the user could have satisfied.
  - Otherwise the user is prompted for the unsatisfied grant controls, in a fixed
    documented order: **MFA → compliant device → hybrid joined → approved client app →
    app protection policy → password change → terms of use → custom controls**.
  - Once all grants are satisfied, session controls apply.

Let's simulate what happens when you forget to exclude the break-glass accounts.


In [ ]:
# A miniature of the documented two-phase evaluation.
GRANT_CONTROL_ORDER = ['mfa', 'compliant_device', 'hybrid_joined', 'approved_app',
                       'app_protection', 'password_change', 'terms_of_use']

# The MFA provider is down: nobody can satisfy MFA, so nobody has any control satisfied.
SATISFIED = {'alice': set(), 'break-glass-1': set()}

def applies_to(user, policy):
    return user not in policy['assignments']['users'].get('exclude', [])

def evaluate(user, policies):
    """Return (decision, detail)."""
    # Phase 1 includes report-only policies; phase 2 (enforcement) does not.
    applicable = [p for p in policies if p['state'] == 'enabled' and applies_to(user, p)]

    # Phase 2, first step: an explicit block ends it. No grant can rescue the session.
    for p in applicable:
        if 'block' in p['grant']['controls']:
            return 'BLOCKED', f'blocked by "{p["name"]}"'

    # Phase 2, second step: the union of every applicable policy's grant controls must
    # be satisfied, and the user is prompted for the unmet ones in the documented order.
    required = {c for p in applicable for c in p['grant']['controls']}
    unmet = [c for c in GRANT_CONTROL_ORDER if c in required and c not in SATISFIED[user]]
    if unmet:
        return 'INTERRUPTED', f'must still satisfy {unmet} (prompted in that order)'
    return 'ALLOWED', 'every applicable policy satisfied'

def report(label, policies):
    print(label)
    for u in ('alice', 'break-glass-1'):
        decision, detail = evaluate(u, policies)
        print(f'  {u:<14} -> {decision:<12} ({detail})')

# Scenario A: break-glass is NOT excluded.
bad_policies = [{
    'name': 'Require MFA for admins',
    'state': 'enabled',
    'assignments': {'users': {'include': 'all', 'exclude': []}, 'apps': {'include': 'all'}},
    'grant': {'operator': 'AND', 'controls': ['mfa']},
}]
print('--- MFA provider is DOWN ---')
report('Policy misconfigured (break-glass NOT excluded):', bad_policies)
print('   !! Nobody can sign in - the tenant is locked. !!\n')

# Scenario B: break-glass IS excluded.
good_policies = [dict(
    bad_policies[0],
    assignments={'users': {'include': 'all', 'exclude': ['break-glass-1']}, 'apps': {'include': 'all'}},
)]
report('Policy correct (break-glass excluded):', good_policies)
print('   OK - ops can sign in with break-glass and fix the broken policy.\n')

assert evaluate('break-glass-1', bad_policies)[0] != 'ALLOWED', \
    'the misconfigured policy must actually lock the break-glass account out'
assert evaluate('break-glass-1', good_policies)[0] == 'ALLOWED', \
    'the exclusion is the whole point of a break-glass account'
assert evaluate('alice', good_policies)[0] == 'INTERRUPTED', \
    'alice is still stuck behind MFA - the exclusion is only for break-glass'

# Scenario C: policies disagree. Alice has now completed MFA, so the grant policy is
# satisfied - but a second policy blocks her anyway.
SATISFIED['alice'].add('mfa')
print('--- Two policies disagree (alice HAS satisfied MFA) ---')
conflicting = good_policies + [{
    'name': 'Block legacy authentication',
    'state': 'enabled',
    'assignments': {'users': {'include': 'all', 'exclude': []}, 'apps': {'include': 'all'}},
    'grant': {'operator': 'OR', 'controls': ['block']},
}]
print(f'  alice with grant policy only -> {evaluate("alice", good_policies)}')
print(f'  alice with grant + block      -> {evaluate("alice", conflicting)}')

assert evaluate('alice', good_policies)[0] == 'ALLOWED', 'MFA satisfied, so the grant policy passes'
assert evaluate('alice', conflicting)[0] == 'BLOCKED', \
    'block must beat grant, even a grant the user already satisfied'

# Report-only never enforces - that is the entire safety property of report-only mode.
report_only = [dict(conflicting[1], state='report-only')]
assert evaluate('alice', report_only)[0] == 'ALLOWED', \
    'a report-only block policy must not block anyone'
print('\nAll Conditional Access evaluation invariants hold.')


## Named locations

Location-based policies need a **named location** defined first:

```bash
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/identity/conditionalAccess/namedLocations' \
  --body '{
    "@odata.type": "#microsoft.graph.ipNamedLocation",
    "displayName": "Corporate offices",
    "isTrusted": true,
    "ipRanges": [
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "203.0.113.0/24"},
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "198.51.100.0/24"}
    ]
  }'
```

## Microsoft Entra ID Protection - risk signals

Conditional Access policies can react to **risk** signals from Microsoft Entra ID Protection:

| Risk type | Question it answers | Examples of signals |
|-----------|--------------------|---------------------|
| **User risk** | Is this *account* compromised? | Leaked credentials (password found in a dump), anomalous user behaviour |
| **Sign-in risk** | Is *this login* suspicious? | Impossible travel, unfamiliar sign-in properties, anonymous IP (Tor), malware-linked IP |

The two risk types pair with different grant controls, and mixing them up is the classic
mistake:

| If... | Then... | Why |
|-------|---------|-----|
| **User risk** = *high* | Force **password reset** | The credential itself is burned; a new one is the only fix |
| **Sign-in risk** = *medium/high* | Force **MFA** | The password may be fine; prove it is really them |

**Require password change** is not a free-floating control: the portal only accepts it on a
policy that uses a *user-risk* condition, targets **all resources**, and selects no other
grant control. (The code cell above asserts exactly that.)

## Exam tips

- Multiple policies are **cumulative**: all applicable policies must be satisfied, and all
  assignments within a policy are ANDed.
- **Block beats grant.** If any applicable policy blocks, enforcement stops immediately.
- Unsatisfied grant controls are prompted in a **fixed order**: MFA, compliant device,
  hybrid joined, approved client app, app protection policy, password change, terms of use.
- CA is evaluated **after** first-factor authentication, and it **cannot grant** access
  that RBAC does not already permit — it can only add requirements or take access away.
- Policies that target **roles or groups** are only evaluated when a token is issued, so a
  user who already holds a token is not covered until it is refreshed. That is why PIM's
  "on activation, require MFA" setting exists.
- Always start a new policy in **report-only** mode. It runs in phase 1 only, so it logs
  what it *would* have done without enforcing. Review sign-in logs, then enable.
- **Break-glass** accounts: excluded from *all* CA policies, alert on every sign-in.
- Licensing: Conditional Access needs **Microsoft Entra ID P1**. Risk-based conditions
  (user risk, sign-in risk) need **Microsoft Entra ID Protection**, which is **P2**. PIM
  needs **P2 or a Microsoft Entra ID Governance** licence. Note the SKU names dropped the
  word "Premium" with the Entra rename — "Azure AD Premium P2" is the old name for
  Microsoft Entra ID P2.

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **PIM eligible vs active** | Eligible = must activate (MFA + justification + optional approval). Active = holds it now. Both can be permanent or time-bound. |
| **PIM activation** | Capped at the role's maximum activation duration (1-24h, default 8h). |
| **CA structure** | Assignments -> Grant controls -> Session controls |
| **CA evaluation** | Phase 1 collect, phase 2 enforce. All applicable policies must pass; block wins; grants prompted in a fixed order. |
| **Named locations** | Define trusted IPs before using location-based policies |
| **Report-only mode** | Phase 1 only. Always test before enforcing. |
| **Break-glass accounts** | Exclude from *all* CA policies; alert on any sign-in |
| **ID Protection** | User risk -> password reset. Sign-in risk -> MFA. Do not swap them. |


---
## ✅ Self-check

Answer these before moving on. Answers are in the next cell.

1. Alice is *permanent eligible* for Global Administrator. Bob is *time-bound active* for
   the same role until Friday. Right now, which of them can delete a user, and what does
   each have to do first?
2. Two enabled policies apply to Carol's sign-in: one grants access if she does MFA, one
   blocks her client app. She completes MFA successfully. What happens?
3. A policy requires MFA **and** a compliant device. Carol's device is compliant but she
   has not done MFA. In what order is she prompted, and why does the order matter for
   reading the sign-in logs?
4. You put a "block access from untrusted countries" policy in **report-only** and a user
   signs in from a blocked country. Are they blocked? What shows up in the logs?
5. Your tenant has Microsoft Entra ID P1 only. Which of these can you build: "require MFA
   for admins", "block legacy auth", "require MFA when sign-in risk is high"?
6. You add Dave to the `Azure-Admins` group that a CA policy targets. Dave signed in an
   hour ago. Is he covered by the policy right now?
7. Why is "require password change when sign-in risk is high" not a policy you can save?
8. PIM is configured with a 2-hour maximum for Global Administrator. Alice requests 8
   hours for an overnight migration. What happens, and what is the right fix?


In [ ]:
answers = """
1. BOB can delete a user right now - an ACTIVE assignment needs no action at all, it just
   expires on Friday. ALICE must ACTIVATE first: MFA, justification, and (per the settings
   in this notebook) approval from the security team. Once activated she has exactly the
   same permissions as Bob, for at most the role's maximum activation duration. Permanent
   vs time-bound is about how long the ASSIGNMENT lasts; eligible vs active is about
   whether she has to do something to use it.

2. BLOCKED. In phase 2 an applicable block policy stops enforcement immediately - her
   satisfied MFA is irrelevant. Block always beats grant, and this is why a badly scoped
   block policy is the fastest way to lock a tenant out.

3. MFA first, then compliant device - the documented prompt order is MFA, compliant
   device, hybrid joined, approved client app, app protection policy, password change,
   terms of use. It matters in the logs because you will see an "interrupt" entry showing
   the later controls as failures on the first pass, then a second entry with everything
   satisfied. Those first-pass "failures" are not real failures.

4. NOT blocked. Report-only policies take part in phase 1 (collect session details) but
   never in phase 2 (enforcement). The sign-in log gets a report-only result of
   "would have been blocked" (reportOnlyFailure), which is exactly what you review before
   flipping the policy to enabled.

5. The first two, yes - Conditional Access itself is a P1 feature. The third, no: sign-in
   risk and user risk conditions come from Microsoft Entra ID Protection, which is P2.
   (PIM also needs P2, or a Microsoft Entra ID Governance licence.)

6. NOT YET. Policies that target roles or groups are evaluated when a token is ISSUED.
   Dave's existing token predates the group change, so the policy does not apply
   retroactively; it applies at his next token issuance. The same gap is why PIM has an
   "on activation, require MFA" setting rather than relying on a CA policy that targets
   the role.

7. Because "Require password change" is only accepted with a USER-risk condition. It also
   has to target all resources and be the only grant control selected. Sign-in risk means
   "this login looks odd" - the answer to that is MFA. User risk means "this credential is
   burned" - the answer to that is a new credential.

8. DENIED - the request cannot exceed the role's maximum activation duration, so the
   portal slider simply will not go past 2 hours. The fix is NOT to widen the maximum to
   8 hours for everyone. Either activate again when it expires (PIM will prompt), or, for
   genuinely long unattended work, use a service principal or managed identity with a
   scoped role instead of a human's Global Admin.
"""
print(answers)


**Next**: [Notebook 3 - App registrations and managed identities](03_app_registrations_and_managed_identities.ipynb)
